# Project Milestone Two: Modeling and Feature Engineering

### Overview

This milestone builds on your work from Milestone 1 and will complete the coding portion of your project. You will:

1. Pick 3 modeling algorithms from those we have studied.
2. Evaluate baseline models using default settings.
3. Engineer new features and re-evaluate models.
4. Use feature selection techniques and re-evaluate.
5. Fine-tune for optimal performance.
6. Select your best model and report on your results. 

You must do all work in this notebook and upload to your team leader's account in Gradescope. There is no
Individual Assessment for this Milestone. 


In [2]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor

# Progress Tracking

#from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))

random_seed = 1

### Prelude: Load your Preprocessed Dataset from Milestone 1

In Milestone 1, you handled missing values, encoded categorical features, and explored your data. Before you begin this milestone, you’ll need to load that cleaned dataset and prepare it for modeling. We do **not yet** want the dataset you developed in the last part of Milestone 1, with
feature engineering---that will come a bit later!

Here’s what to do:

1. Return to your Milestone 1 notebook and rerun your code through Part 3, where your dataset was fully cleaned (assume it’s called `df_cleaned`).

2. **Save** the cleaned dataset to a file by running:

>   df_cleaned.to_csv("zillow_cleaned.csv", index=False)

3. Switch to this notebook and **load** the saved data:

>   df = pd.read_csv("zillow_cleaned.csv")

4. Create a **train/test split** using `train_test_split`.  
   
6. **Standardize** the features (but not the target!) using **only the training data.** This ensures consistency across models without introducing data leakage from the test set:

>   scaler = StandardScaler()   
>   X_train_scaled = scaler.fit_transform(X_train)    
  
**Notes:** 

- You will have to redo the scaling step if you introduce new features (which have to be scaled as well).


In [10]:
# 1.2.3. Load cleaned data as df:

df = pd.read_csv('zillow_cleaned.csv') # using latest csv

In [11]:
# Check df:

df.head()

,baths_calc,baths_full,baths_three_quarter,bedroomcnt,building_quality,fireplaces,garage_spaces,garage_sqft,has_deck,has_hot_tub,...,latitude,longitude,lot_size,sqft,stories,tax_value,total_rooms,units,year_built,zip_code
0,3.5,3.0,1.0,4.0,8.0,0,2.0,633.0,0,0,...,33634931.0,-117869207.0,4506.0,3100.0,1.0,1023282.0,0.0,1.0,1998.0,96978.0
1,1.0,1.0,0.0,2.0,9.0,1,1.0,0.0,0,0,...,34449266.0,-119281531.0,12647.0,1465.0,1.0,464000.0,5.0,1.0,1967.0,97099.0
2,2.0,2.0,0.0,3.0,7.0,0,2.0,440.0,0,0,...,33886168.0,-117823170.0,8432.0,1243.0,1.0,564778.0,6.0,1.0,1962.0,97078.0
3,3.0,3.0,0.0,4.0,8.0,0,0.0,0.0,0,0,...,34245180.0,-118240722.0,13038.0,2376.0,1.0,145143.0,0.0,1.0,1970.0,96330.0
4,3.0,3.0,0.0,3.0,8.0,0,0.0,0.0,0,0,...,34185120.0,-118414640.0,278581.0,1312.0,1.0,119407.0,0.0,1.0,1964.0,96451.0


In [12]:
df.describe()

,baths_calc,baths_full,baths_three_quarter,bedroomcnt,building_quality,fireplaces,garage_spaces,garage_sqft,has_deck,has_hot_tub,...,latitude,longitude,lot_size,sqft,stories,tax_value,total_rooms,units,year_built,zip_code
count,77377.000000,77377.000000,77377.000000,77377.000000,77377.000000,77377.00000,77377.000000,77377.000000,77377.000000,77377.000000,...,7.737700e+04,7.737700e+04,7.737700e+04,77377.000000,77377.000000,7.737700e+04,77377.000000,77377.000000,77377.000000,77377.000000
mean,2.304199,2.237797,0.131848,3.060832,6.810318,0.12978,0.598666,115.439950,0.007935,0.019890,...,3.400868e+07,-1.182039e+08,2.756849e+04,1785.647776,1.098763,4.888552e+05,1.479445,1.072282,1968.598188,96586.338692
std,0.991370,0.978037,0.342881,1.131744,1.564537,0.40390,0.917890,222.846065,0.088726,0.139622,...,2.652085e+05,3.591728e+05,1.166800e+05,956.088804,0.317119,6.501854e+05,2.825907,0.948522,23.798031,3797.230312
min,0.000000,0.000000,0.000000,0.000000,1.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,3.333953e+07,-1.194754e+08,2.360000e+02,128.000000,1.000000,1.000000e+03,0.000000,1.000000,1824.000000,95982.000000
25%,2.000000,2.000000,0.000000,2.000000,6.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,3.381496e+07,-1.184153e+08,5.603000e+03,1182.000000,1.000000,2.069580e+05,0.000000,1.000000,1953.000000,96193.000000
50%,2.000000,2.000000,0.000000,3.000000,7.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,3.402244e+07,-1.181810e+08,7.085000e+03,1542.000000,1.000000,3.588780e+05,0.000000,1.000000,1970.000000,96389.000000
75%,3.000000,3.000000,0.000000,4.000000,8.000000,0.00000,2.000000,0.000000,0.000000,0.000000,...,3.417451e+07,-1.179292e+08,1.077300e+04,2113.000000,1.000000,5.685400e+05,0.000000,1.000000,1987.000000,96987.000000
max,18.000000,18.000000,7.000000,16.000000,12.000000,5.00000,14.000000,4251.000000,1.000000,1.000000,...,3.481877e+07,-1.175546e+08,6.971010e+06,35640.000000,6.000000,4.906124e+07,15.000000,237.000000,2016.000000,399675.000000


In [13]:
## we can take the log before train test split as it does not depend on values from the dataset. 

df['sqft_log'] = np.log(df['sqft'])

df['tax_value_log'] = np.log(df['tax_value'])

df['lot_size_log'] = np.log(df['lot_size'])

df = df.drop(columns = ['sqft','tax_value','lot_size'])



## drop id_city and zip code unless we want to one hot them 

df = df.drop(columns = ['id_city','zip_code'])



In [16]:
# 4. Create train/test split:

X = df.drop(columns=['tax_value_log'])
y = df['tax_value_log']

X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2, 
                                                    random_state = random_seed)

In [17]:
# 5. Standardize features (not target, not testing data) using standard scalar:

X_train_scaled = StandardScaler().fit_transform(X_train)

# X_train_scaled is now ready to be used in the models
# X_test will have to be scaled later

### Part 1: Picking Three Models and Establishing Baselines [6 pts]

Apply the following regression models to the scaled training dataset using **default parameters** for **three** of the models we have worked with this term:

- Linear Regression
- Ridge Regression
- Lasso Regression
- Decision Tree Regression
- Bagging
- Random Forest
- Gradient Boosting Trees

For each of the three models:
- Use **repeated cross-validation** (e.g., 5 folds, 5 repeats).
- Report the **mean and standard deviation of CV MAE Score**. 


In [21]:
# Model A: Lasso

# Gaurav model here

from sklearn.linear_model import Lasso
from sklearn.model_selection import RepeatedKFold, cross_val_score
import numpy as np

lasso = Lasso(random_state=random_seed)

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_seed)

scores = cross_val_score(
    lasso,
    X_train_scaled,
    y_train,
    cv=cv,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

rmse_scores = np.sqrt(-scores)

print("RMSE scores:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())
print("Std RMSE:", rmse_scores.std())

RMSE scores: [0.8742972  0.87061369 0.85769361 0.8640924  0.87486617 0.87190447
 0.86368869 0.87530152 0.86320646 0.8674538  0.86255741 0.87135211
 0.87005571 0.88103411 0.85647533 0.86557241 0.87127419 0.85819376
 0.87812713 0.86847141 0.87670478 0.87720068 0.85240733 0.87245105
 0.86259085]
Mean RMSE: 0.8683034511613913
Std RMSE: 0.007319057528565828


In [ ]:
# Model B: Decision Tree

## Josh model here


from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.tree import DecisionTreeRegressor
import numpy as np

tree = DecisionTreeRegressor(
    random_state=random_seed
)

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_seed)

scores = cross_val_score(
    tree,
    X_train_scaled,
    y_train,
    cv=cv,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

rmse_scores = np.sqrt(-scores)

print("RMSE scores:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())
print("Std RMSE:", rmse_scores.std())



RMSE scores: [0.88733548 0.88835927 0.90282721 0.89367394 0.89445655 0.89145858
 0.89263594 0.88468305 0.88551347 0.8828414  0.89698519 0.87768246
 0.8973613  0.9049879  0.87635239 0.89364202 0.8886696  0.88841848
 0.89344424 0.90073097 0.89499274 0.90333533 0.88192134 0.88853611
 0.88120044]
Mean RMSE: 0.8908818162715619
Std RMSE: 0.007628147501039563


In [20]:
# Model C: Gradient Boosting Trees

# Jessica model here

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import RepeatedKFold, cross_val_score
import numpy as np

gbr = GradientBoostingRegressor(random_state=random_seed)

cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_seed)

scores = cross_val_score(
    gbr,
    X_train_scaled,
    y_train,
    cv=cv,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

rmse_scores = np.sqrt(-scores)

print("RMSE scores:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())
print("Std RMSE:", rmse_scores.std())



RMSE scores: [0.64500049 0.63808364 0.64421577 0.64079993 0.64160987 0.64356202
 0.63797809 0.64388052 0.63581665 0.64464332 0.6398086  0.64659655
 0.64264523 0.64536278 0.63388065 0.642724   0.63923203 0.63771521
 0.64400477 0.64639537 0.64976011 0.64716699 0.63110887 0.64445397
 0.63703824]
Mean RMSE: 0.6417393460393067
Std RMSE: 0.004409783087175823


### Part 1: Discussion [3 pts]

In a paragraph or well-organized set of bullet points, briefly compare and discuss:

  - Which model performed best overall?
  - Which was most stable (lowest std)?
  - Any signs of overfitting or underfitting?

> Your text here

### Part 2: Feature Engineering [6 pts]

Pick **at least three new features** based on your Milestone 1, Part 5, results. You may pick new ones or
use the same ones you chose for Milestone 1. 

Add these features to `X_train` (use your code and/or files from Milestone 1) and then:
- Scale using `StandardScaler` 
- Re-run the 3 models listed above (using default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [ ]:
# 3 engineered features saved to df:

In [ ]:
# Model A using engineered features:

In [ ]:
# Model B using engineered features:

In [ ]:
# Model C using engineered features:

### Part 2: Discussion [3 pts]

Reflect on the impact of your new features:

- Did any models show notable improvement in performance?

- Which new features seemed to help — and in which models?

- Do you have any hypotheses about why a particular feature helped (or didn’t)?




> Your text here

### Part 3: Feature Selection [6 pts]

Using the full set of features (original + engineered):
- Apply **feature selection** methods to investigate whether you can improve performance.
  - You may use forward selection, backward selection, or feature importance from tree-based models.
- For each model, identify the **best-performing subset of features**.
- Re-run each model using only those features (with default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [ ]:
# Feature selection for model A:


In [ ]:
# Model A using best features:

In [ ]:
# Model B using best features:

In [ ]:
# Model C using best features:

### Part 3: Discussion [3 pts]

Analyze the effect of feature selection on your models:

- Did performance improve for any models after reducing the number of features?

- Which features were consistently retained across models?

- Were any of your newly engineered features selected as important?


> Your text here

### Part 4: Fine-Tuning Your Three Models [6 pts]

In this final phase of Milestone 2, you’ll select and refine your **three most promising models and their corresponding data pipelines** based on everything you've done so far, and pick a winner!

1. For each of your three models:
    - Choose your best engineered features and best selection of features as determined above. 
   - Perform hyperparameter tuning using `sweep_parameters`, `GridSearchCV`, `RandomizedSearchCV`, `Optuna`, etc. as you have practiced in previous homeworks. 
3. Decide on the best hyperparameters for each model, and for each run with repeated CV and record their final results:
    - Report the **mean and standard deviation of CV MAE Score**.  

In [ ]:
# Hyperparamtere tuning for Model A:


In [ ]:
# Hyperparamtere tuning for Model B:


In [ ]:
# Hyperparamtere tuning for Model C:


### Part 4: Discussion [3 pts]

Reflect on your tuning process and final results:

- What was your tuning strategy for each model? Why did you choose those hyperparameters?
- Did you find that certain types of preprocessing or feature engineering worked better with specific models?


> Your text here

### Part 5: Final Model and Design Reassessment [6 pts]

In this part, you will finalize your best-performing model.  You’ll also consolidate and present the key code used to run your model on the preprocessed dataset.
**Requirements:**

- Decide one your final model among the three contestants. 

- Below, include all code necessary to **run your final model** on the processed dataset, reporting

    - Mean and standard deviation of CV MAE Score.
    
    - Test score on held-out test set. 




In [ ]:
# Rerun of best model with engineered features, best features, and hyperparamter tuning on training data:

In [ ]:
# Model performance on test data:

### Part 5: Discussion [8 pts]

In this final step, your goal is to synthesize your entire modeling process and assess how your earlier decisions influenced the outcome. Please address the following:

1. Model Selection:
- Clearly state which model you selected as your final model and why.

- What metrics or observations led you to this decision?

- Were there trade-offs (e.g., interpretability vs. performance) that influenced your choice?

2. Revisiting an Early Decision

- Identify one specific preprocessing or feature engineering decision from Milestone 1 (e.g., how you handled missing values, how you scaled or encoded a variable, or whether you created interaction or polynomial terms).

- Explain the rationale for that decision at the time: What were you hoping it would achieve?

- Now that you've seen the full modeling pipeline and final results, reflect on whether this step helped or hindered performance. Did you keep it, modify it, or remove it?

- Justify your final decision with evidence—such as validation scores, visualizations, or model diagnostics.

3. Lessons Learned

- What insights did you gain about your dataset or your modeling process through this end-to-end workflow?

- If you had more time or data, what would you explore next?

> Your text here